<a href="https://colab.research.google.com/github/Cheikhcisse1/projet_FN/blob/main/SerieTemporelle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Les bibiothéques Besion

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss
from scipy.special import inv_boxcox

In [ ]:
class SerieTemporelle:
  def __init__(self, donnee):
    self.donnee = donnee.copy()
    if 'date' in self.donnee.columns:
      self.donnee['date'] = pd.to_datetime(self.donnee['date'])
      self.donnee = self.donnee.sort_values('date')

  def traitementDes_donne(self):
    print("Aperçu")
    print(self.donnee.head())
    print('-----------------------'*5)
    print("Shape")
    print(self.donnee.shape)
    print('-----------------------'*5)
    print("Description")
    print(self.donnee.describe())
    print('-----------------------'*5)
    print("Info")
    print(self.donnee.info())
    print('-----------------------'*5)
    print("Valeurs manquantes")
    print(self.donnee.isnull().sum())
    print('-----------------------'*5)
    print("Valeurs dupliqués")
    print(self.donnee.duplicated().sum())
    print('-----------------------'*5)

  def visualisation(self):
    plt.figure(figsize=(10, 5))
    if 'date' in self.donnee.columns:
      plt.plot(self.donnee['date'], self.donnee['aqi'], label="AQI")
    else:
      plt.plot(self.donnee.index, self.donnee['aqi'], label="AQI")
    plt.title("L'évolution de l'AQI mensuel")
    plt.xlabel("Date")
    plt.ylabel("Valeurs AQI")
    plt.legend()
    plt.grid(True)
    plt.show()

  def traitementValeurmanquantes(self):
    # Interpolation linéaire pour maintenir la continuité temporelle
    self.donnee['aqi'] = self.donnee['aqi'].interpolate(method='linear')
    print("Valeurs manquantes après traitement (interpolation)")
    print(self.donnee.isnull().sum())
    print('-----------------------'*5)
    print("Valeurs dupliqués après traitement")
    print(self.donnee.duplicated().sum())
    print('-----------------------'*5)
    print("Shape après traitement")
    print(self.donnee.shape)
    print('-----------------------'*5)

  def preparer_index(self):
    # Méthode interne pour s'assurer que l'index est temporel et régulier
    if 'date' in self.donnee.columns:
      self.donnee.set_index('date', inplace=True)
    if not isinstance(self.donnee.index, pd.DatetimeIndex):
      self.donnee.index = pd.to_datetime(self.donnee.index)
    # .to_timestamp() sans argument replace correctement l'index au début du mois
    self.donnee.index = self.donnee.index.to_period('M').to_timestamp()

  def decomposition(self):
    self.preparer_index()
    # Décomposition additive avec une période de 12 (mensuelle)
    resultat = seasonal_decompose(self.donnee['aqi'], model='additive', period=12)

    # Visualisation
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

    ax1.plot(self.donnee['aqi'], label='Original')
    ax1.legend(loc='upper left')
    ax1.set_title('Décomposition de la série temporelle')
    ax1.grid(True)

    ax2.plot(resultat.trend, label='Trend', color='orange')
    ax2.legend(loc='upper left')
    ax2.grid(True)

    ax3.plot(resultat.seasonal, label='Seasonal', color='green')
    ax3.legend(loc='upper left')
    ax3.grid(True)

    ax4.plot(resultat.resid, label='Residual', color='red')
    ax4.legend(loc='upper left')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    decomposition_df = pd.DataFrame({
        'aqi': self.donnee['aqi'],
        'trend': resultat.trend,
        'seasonal': resultat.seasonal,
        'residual': resultat.resid
    }, index=self.donnee.index)
    return decomposition_df

  def testSaisonnalité(self):
    self.preparer_index()
    resultat = seasonal_decompose(self.donnee['aqi'], model='additive', period=12)

    plt.figure(figsize=(10, 4))
    plt.plot(resultat.seasonal, label='Seasonal', color='green')
    plt.title('Composante Saisonnière (Saisonnalité)')
    plt.xlabel('Date')
    plt.ylabel('Valeur AQI')
    plt.legend()
    plt.grid(True)
    plt.show()

    decomposition_df = pd.DataFrame({
        'aqi': self.donnee['aqi'],
        'trend': resultat.trend,
        'seasonal': resultat.seasonal,
        'residual': resultat.resid
    }, index=self.donnee.index)
    return decomposition_df

  def ADF_test(self):
    result = adfuller(self.donnee['aqi'])
    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    # test
    if result[1] > 0.05:
      print('La série n\'est pas stationnaire')
    else:
      print('La série est stationnaire')

    for key, value in result[4].items():
        print('\t%s: %.3f' % (key, value))

    plt.figure(figsize=(10, 4))
    plot_acf(self.donnee['aqi'], lags=40)
    plt.title("test ADF")
    plt.show()


  def KPSS_test(self):
    result = kpss(self.donnee['aqi'])
    print('KPSS Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    # test
    if result[1] > 0.05:
      print('La série n\'est pas stationnaire')
    else:
      print('La série est stationnaire')

    for key, value in result[3].items():
        print('\t%s: %.3f' % (key, value))

    plt.figure(figsize=(10, 4))
    plot_pacf(self.donnee['aqi'], lags=40)
    plt.title("test KPSS")
    plt.show()

  def courbe_ACF_PACF(self):
    self.preparer_index()
    plt.figure(figsize=(10, 4))
    plot_acf(self.donnee['aqi'], lags=40)
    plt.title('Courbe ACF')
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 4))
    plot_pacf(self.donnee['aqi'], lags=40)
    plt.title('Courbe PACF')
    plt.grid(True)
    plt.show()

  # rendre le stationnaire avec la differenciation
  def differenciation(self):
    self.preparer_index()
    self.donnee['aqi_diff'] = self.donnee['aqi'].diff(12)
    self.donnee.dropna(inplace=True)

    plt.figure(figsize=(10, 4))
    plt.plot(self.donnee['aqi_diff'], label='Differenciée', color='blue')
    plt.title('Differenciation de la série temporelle')
    plt.xlabel('Date')
    plt.ylabel('Valeur AQI')
    plt.legend()
    plt.grid(True)
    plt.show()
    return self.donnee['aqi_diff']

  def logarithme(self):
    self.preparer_index()
    self.donnee['aqi_log'] = np.log(self.donnee['aqi'])

    plt.figure(figsize=(10, 4))
    plt.plot(self.donnee['aqi_log'], label='Logarithmique', color='green')
    plt.title('Logarithme de la série temporelle')
    plt.xlabel('Date')
    plt.ylabel('Valeur AQI')
    plt.legend()
    plt.grid(True)
    plt.show()
    return self.donnee['aqi_log']

  def boxscox(self):
    self.preparer_index()
    self.donnee['aqi_boxcox'], _ = stats.boxcox(self.donnee['aqi'])

    plt.figure(figsize=(10, 4))
    plt.plot(self.donnee['aqi_boxcox'], label='Box-Cox', color='red')
    plt.title('Transformation Box-Cox de la série temporelle')
    plt.xlabel('Date')
    plt.ylabel('Valeur AQI')
    plt.legend()
    plt.grid(True)
    plt.show()
    return self.donnee['aqi_boxcox']

In [ ]:
if __name__ == "__main__":
  chemin = '/content/aqi_mensuel.csv'

  # Étape 1 : Analyse exploratoire & Visualisation initiale
  donnee_base = pd.read_csv(chemin)
  donnee = SerieTemporelle(donnee_base.copy())
  donnee.traitementDes_donne()
  donnee.visualisation()

  # Étape 2 : Traitement des valeurs manquantes
  donnee.traitementValeurmanquantes()

  # Étape 3 : Décomposition et affichage
  print("la décomposition ")
  df_decomp = donnee.decomposition()
  print(df_decomp.head(12))

  print('********************************'*5)
  # Teste de saisonnalité
  print("la saisonnalité")
  df_saison = donnee.testSaisonnalité()
  print(df_saison.head(12))

  print("---------------------------------------------------")
  donnee.ADF_test()
  print("---------------------------------------------------")
  donnee.KPSS_test()

  print("---------------------------------------------------")
  donnee.courbe_ACF_PACF()

In [ ]:
donnee.differenciation()

In [ ]:
donnee.logarithme()

In [ ]:
import scipy.stats as stats
donnee.boxscox()

In [ ]:
!pip install skforecast

In [ ]:
from skforecast.stats import Arima, Sarimax
from pmdarima import auto_arima
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import grid_search_forecaster

class Sarimax_model(SerieTemporelle):
  def __init__(self, donnee, p, d, q, P, D, Q, s):
    super().__init__(donnee)
    self.p = p
    self.d = d
    self.q = q
    self.P = P
    self.D = D
    self.Q = Q
    self.s = s

  def entrainement_sarimax(self):
    self.preparer_index()
    self.model = Sarimax(order=(self.p, self.d, self.q), seasonal_order=(self.P, self.D, self.Q, self.s))
    self.model

  def entrainement_arima(self):
    self.preparer_index()
    self.model = Arima(order=(self.p, self.d, self.q))
    self.model



In [ ]:
import statsmodels.api as sm
import pandas as pd

if __name__ == "__main__":
  # Chargement
  data = sm.datasets.get_rdataset("AirPassengers", "datasets")
  dates = pd.date_range('1949-01', periods=144, freq='MS')
  serie = pd.Series(data.data['value'].values, index=dates, name='passagers')

  # Conversion et renommage correct des deux colonnes pour la classe
  data_df = serie.reset_index()
  data_df.columns = ['date', 'aqi']

  donneee = SerieTemporelle(data_df)
  donneee.traitementDes_donne()
  donneee.visualisation()
  donneee.traitementValeurmanquantes()
  donneee.decomposition()
  donneee.testSaisonnalité()
  donneee.ADF_test()
  donneee.KPSS_test()
  donneee.courbe_ACF_PACF()

In [ ]:
# 1. Application de la transformation de Box-Cox (qui stabilise parfaitement la variance)
donneee.boxscox()

# 2. Application d'une unique différenciation simple (d=1) sur la série Box-Cox
data_stationnaire = donneee.donnee.copy()
data_stationnaire['aqi_boxcox_diff'] = data_stationnaire['aqi_boxcox'].diff(1)
data_stationnaire.dropna(inplace=True)

# Création d'une nouvelle instance pour analyser la série transformée
series_transformee = SerieTemporelle(data_stationnaire.reset_index()[['date', 'aqi_boxcox_diff']].rename(columns={'aqi_boxcox_diff': 'aqi'}))

# 3. Validation visuelle et statistique de la stationnarité par les deux tests
print("=== ANALYSE DE LA SÉRIE TRANSFORMÉE (BOX-COX + DIFFÉRENCIATION SIMPLE) ===")
series_transformee.visualisation()
print("--- TEST ADF ---")
series_transformee.ADF_test()
print("\n--- TEST KPSS ---")
series_transformee.KPSS_test()
print("\n--- COURBES ACF / PACF ---")
series_transformee.courbe_ACF_PACF()

In [ ]:
# Affichage direct des statistiques et p-value du KPSS pour la série transformée
from statsmodels.tsa.stattools import kpss
stat, p_value, lags, crit = kpss(series_transformee.donnee['aqi'])
print(f"Statistique KPSS : {stat:.6f}")
print(f"p-value KPSS : {p_value:.6f}")
if p_value > 0.05:
    print("Le test KPSS confirme que la série EST stationnaire (p-value > 0.05)")
else:
    print("Le test KPSS indique que la série n'est TOUJOURS PAS stationnaire (p-value <= 0.05)")